# 01 · Silver – reference tables
**Phase 4 – Silver layer.** Cleans the small reference tables `cities` and `cuisine_category_translation` from Bronze and saves them to `glovo_dev.silver`.

**Rules for Silver**
- Correct types, consistent values, no duplicates.
- Bad rows are not deleted: they go to `glovo_dev.silver.quarantine` with a reason.
- Audit columns from Bronze are kept, so every row can still be traced to its source file.

## 1. Settings

In [0]:
from pyspark.sql import functions as F
CATALOG = "glovo_dev"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

## 2. Look at the Bronze data

In [0]:
cities_bronze = spark.table(f"{BRONZE}.cities")
display(cities_bronze)

## 3. City match key
Imported from the shared module `src/glovo/silver/cleaning.py` (unit-tested in CI), so every Silver notebook uses the same logic.
`" ŁÓDŹ "`, `"Lodz"` and `"Łódź"` all become `"lodz"`.

In [0]:
import os
import sys

sys.path.append(os.path.abspath("../../src"))
from glovo.silver.cleaning import city_key

# quick test
test = spark.createDataFrame([(" ŁÓDŹ ",), ("Lodz",), ("łódź",), ("KRAKÓW",)], ["city"])
display(test.withColumn("key", city_key(F.col("city"))))

## 4. Silver cities
Keeps the official city list, adds `city_key` for matching, and checks that every key is unique.

In [0]:
cities_silver = (
    cities_bronze
    .withColumn("city_name", F.trim(F.col("city_name")))
    .withColumn("city_key", city_key(F.col("city_name")))
    .select(
        "city_id", "city_name", "city_key", "voivodeship", "population",
        "latitude", "longitude", "city_tier",
        "_ingested_at", "_source_file",
    )
)

display(cities_silver)

In [0]:
total = cities_silver.count()
unique_keys = cities_silver.select("city_key").distinct().count()
print(f"rows: {total}, unique keys: {unique_keys}")

assert total == unique_keys, "Two cities have the same key!"

## Saving the Silver Cities Table

In [0]:
(
    cities_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.cities")

)

In [0]:
%sql
SELECT city_name, city_key, city_tier
FROM glovo_dev.silver.cities
ORDER BY population DESC
LIMIT 10

## 5. Silver cuisine categories
Cleans the cuisine codes, adds a readable name for reports, and checks that each code is unique.

In [0]:
cuisine_bronze = spark.table(f"{BRONZE}.cuisine_category_translation")
display(cuisine_bronze)

## Data Quality Check: Validate Uniqueness and Completeness of Cuisine Category Translations

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT cuisine_category_pl) AS unique_pl,
    COUNT(DISTINCT cuisine_category_en) AS unique_en,

    COUNT_IF(cuisine_category_pl IS NULL) AS null_pl,
    COUNT_IF(cuisine_category_en IS NULL) AS null_en,

    CASE
        WHEN COUNT(*) = COUNT(DISTINCT cuisine_category_pl)
         AND COUNT(*) = COUNT(DISTINCT cuisine_category_en)
         AND COUNT_IF(cuisine_category_pl IS NULL) = 0
         AND COUNT_IF(cuisine_category_en IS NULL) = 0
        THEN 'PASS'
        ELSE 'FAIL: Duplicate or NULL cuisine categories!'
    END AS check_result

FROM glovo_dev.bronze.cuisine_category_translation;

In [0]:
cuisine_silver = (
    cuisine_bronze
    .withColumn("cuisine_category_pl", F.lower(F.trim(F.col("cuisine_category_pl"))))
    .withColumn("cuisine_category_en", F.lower(F.trim(F.col("cuisine_category_en"))))
    .withColumn("cuisine_name", F.initcap(F.regexp_replace(F.col("cuisine_category_en"), "_", " ")))
    .select("cuisine_category_pl", "cuisine_category_en", "cuisine_name", "_ingested_at", "_source_file")
)

display(cuisine_silver)

total = cuisine_silver.count()
unique_codes = cuisine_silver.select("cuisine_category_pl").distinct().count()
print(f"rows: {total}, unique codes: {unique_codes}")
assert total == unique_codes, "Duplicate cuisine codes!"

In [0]:
(
    cuisine_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.cuisine_categories")

)

## 6. Result

In [0]:
%sql
SELECT 'cities' AS table_name, COUNT(*) AS row_count FROM glovo_dev.silver.cities
UNION ALL
SELECT 'cuisine_categories', COUNT(*) FROM glovo_dev.silver.cuisine_categories